# 02 · Factorial fraccionado $2^{k-1}$ y alias (Python)

**Semana 3 — Diseños $2^k$ y fraccionados.**

## Objetivos
- Construir una **media fracción** $2^{4-1}$ con relación de definición $I=ABCD$.
- Entender la **estructura de alias** y cómo interpretar los efectos confundidos.

> Teoría: [`../teoria/03-factoriales-fraccionados.md`](../teoria/03-factoriales-fraccionados.md).

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

np.random.seed(3)

## 1. De $2^4$ a media fracción $2^{4-1}$

Partimos del mismo experimento de filtración. Seleccionamos la **media fracción** con $ABCD = +1$ (relación de definición $I=ABCD$): 8 de las 16 corridas.

In [ ]:
df = pd.read_csv('../datos/filtracion-2k.csv')
df['ABCD'] = df.A * df.B * df.C * df.D
half = df[df.ABCD == 1].drop(columns='ABCD').reset_index(drop=True)
half

## 2. Estructura de alias

Con $I=ABCD$, multiplicando cada efecto por la relación de definición:
$$A=BCD,\quad B=ACD,\quad C=ABD,\quad D=ABC,$$
$$AB=CD,\quad AC=BD,\quad AD=BC.$$
Cada columna estima la **suma** de su par de alias. Con 8 corridas solo podemos ajustar 7 efectos (3 principales visibles + interacciones).

In [ ]:
# Ajuste sobre A, B, C y sus interacciones (D = ABC queda implícito por el alias)
modelo = smf.ols('tasa ~ A*B*C', data=half).fit()
ef = (modelo.params * 2).drop('Intercept')
alias = {'A':'A+BCD','B':'B+ACD','C':'C+ABD',
         'A:B':'AB+CD','A:C':'AC+BD','B:C':'BC+AD','A:B:C':'ABC+D'}
tabla = pd.DataFrame({'columna': ef.index,
                      'estima (alias)': [alias[i] for i in ef.index],
                      'efecto': ef.values.round(2)})
tabla

## 3. Interpretación

Comparando con el análisis del $2^4$ completo (notebook 01), donde los grandes efectos eran $A\approx21.6$, $AC\approx-18.1$, $AD\approx16.6$, $D\approx14.6$, $C\approx9.9$:

- La columna **$A$** ($\approx19$) recoge $A+BCD\approx A$ (BCD es ruido). ✅
- La columna **$AC$** ($\approx-18.5$) recoge $AC+BD\approx AC$. ✅
- La columna **$BC$** ($\approx19$) recoge $BC+AD\approx AD$: ¡el gran efecto $AD$ aparece confundido en la columna $BC$! ⚠️
- La columna **$ABC$** ($\approx16.5$) recoge $D+ABC\approx D$. ✅

Con 8 corridas (la mitad) **recuperamos casi la misma información** sobre los efectos dominantes, pero **no podemos separar** $AC$ de $BD$ ni $AD$ de $BC$. Para romper estos alias se usaría un **fold-over** (teoría, §6).

## 4. Conclusión
- La media fracción $2^{4-1}$ con $I=ABCD$ es de **resolución IV**: efectos principales limpios de interacciones de dos factores; las de dos factores se confunden entre sí.
- Ahorra la mitad de las corridas a costa de ambigüedad en las interacciones dobles.
- Estrategia secuencial: cribar con la fracción, luego **aumentar** para resolver alias.

> Equivalente en R: [`02-fraccionado_r.ipynb`](02-fraccionado_r.ipynb).